# L2-Thinker regional vocabulary trim: full pipeline (Colab, A100)

Reproduces every number in the project write-up from scratch: GGUF conversion of `CohereLabs/tiny-aya-l2-thinker`, the South Asia selection corpus and keep file, the trimmed GGUF, tokenization drift, size and CPU speed, paired greedy generation (trimmed vs same-session control), the readout, divergence replay, and the KV-cache probe.

**Budget.** About 10 GPU hours on an A100-40GB for round 1 and round 2 together, plus ~40 min of conversion the first time. Everything is resumable: generation rows are written one at a time and mirrored to Drive, so a disconnect costs minutes, not hours.

**Prerequisites.** A Hugging Face token (Colab secret `HF_TOKEN`) with the L2-Thinker licence accepted on the model page (CC-BY-NC, research only). Google Drive mounted for the mirror. The harness repo `ayastaga/tinyaya-quant-safety` must contain `tinyaya_eval/l2.py` (the version in this repo's `vendor/`) and `tinyaya_eval/vocabtrim.py`.

**Two ways to run it.**
1. *Verify the shipped keep file* (recommended): skip Part 11, copy `keep/l2thinker_trim_keep.json` into `calib/trim/l2thinker/` in Part 10b, and let Part 12 rebuild the trimmed GGUF from it. The trimmed model is then byte-for-byte the one evaluated in the write-up.
2. *Rebuild the keep file*: run Part 11 (streams ~650 MB of corpus from the Hub, ~20 min). Corpus streams are not perfectly stable over time, so the keep set may differ by a few hundred tokens; the pre-registered criteria should still hold.

**Session rules that cost hours when ignored** (from the project handoff): interrupting a runner cell kills its jobs, so watch progress from Colab's Terminal; a disconnect wipes `/content` but not Drive; paired comparisons must come from one GPU type, never mix T4 and A100 rows; never start a runner while old `gen` processes live.

## 0 · Toolchain

Same pins as the safety study (llama.cpp `5f436ddd`, llama-cpp-python 0.3.35). Fails cheaply if the GPU runtime, gated access or the pinned wheel is missing. **Restart the session after this part** (Runtime → Restart session); Part 0 is not re-run afterwards.

In [ ]:
REPO_URL = "https://github.com/ayastaga/tinyaya-quant-safety"
REPO_REF = "main"            # pin to a tag for a citable run; must contain tinyaya_eval/l2.py (and vocabtrim.py)
REPO_DIR = "/content/tinyaya-quant-safety"

import subprocess
r = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
assert r.returncode == 0 and r.stdout.strip(), "GPU runtime required"
print(r.stdout.strip())

In [ ]:
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
from huggingface_hub import hf_hub_download
# L2-Thinker is gated separately from tiny-aya-global: accept its licence on the model page first (CC-BY-NC, research only).
hf_hub_download("CohereLabs/tiny-aya-l2-thinker", "config.json", token=os.environ["HF_TOKEN"])
print("HF: gated access to tiny-aya-l2-thinker confirmed")

In [ ]:
import os, subprocess, torch
os.path.exists(REPO_DIR) or subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
!cd {REPO_DIR} && git fetch -q origin && git checkout -q {REPO_REF} && echo "repo @ $(git rev-parse --short HEAD)"
assert os.path.exists(f"{REPO_DIR}/tinyaya_eval/l2.py"), "tinyaya_eval/l2.py missing: copy it into the repo and push (or drop it in now)"

import re
LLAMA_CPP_PIN = re.search(r'LLAMA_CPP_PIN\s*=\s*"(\w+)"', open(f"{REPO_DIR}/tinyaya_eval/config.py").read()).group(1)
os.path.exists("/content/llama.cpp/.git") or subprocess.run(["git", "clone", "-q", "https://github.com/ggml-org/llama.cpp", "/content/llama.cpp"], check=True)
!cd /content/llama.cpp && git fetch -q origin && git checkout -q {LLAMA_CPP_PIN} && echo "llama.cpp @ $(git rev-parse --short HEAD)"
!cd /content/llama.cpp && ( [ -f requirements/requirements-convert_hf_to_gguf.txt ] && pip install -q -r requirements/requirements-convert_hf_to_gguf.txt || pip install -q -r requirements.txt )
!pip install -q -e /content/llama.cpp/gguf-py

cuda = (torch.version.cuda or "12.4").split(".")
cu = f"cu{cuda[0]}{cuda[1]}" if f"cu{cuda[0]}{cuda[1]}" in ("cu121", "cu122", "cu123", "cu124") else "cu124"
!pip install -q llama-cpp-python==0.3.35 --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/{cu}
!pip install -q -e {REPO_DIR}
!pip install -q -U "protobuf>=5.29,<6" fasttext-wheel datasets pandas
!python -m tinyaya_eval.l2 patch --repo {REPO_DIR}
print("install complete")

In [ ]:
import llama_cpp
assert llama_cpp.__version__ == "0.3.35", llama_cpp.__version__
if not llama_cpp.llama_supports_gpu_offload():
    !CMAKE_ARGS="-DGGML_CUDA=on" pip install -q --force-reinstall --no-cache-dir llama-cpp-python==0.3.35
    raise SystemExit("CUDA wheel rebuilt from source; restart the session and re-run this cell")
print("GPU offload: OK")

> **Restart the session** (Runtime → Restart session) before Part 1. Part 0 is not re-run afterwards.

## 1 · Session state

`MODEL = "l2thinker"`. Generation budget is per model here: `TINYAYA_MAX_NEW=8192`, `TINYAYA_N_CTX=10240` (both are folded into every run identity). 32K ctx would cost ≈2.4 GB of KV per stream at f16; 10240 is what these prompts need.

In [ ]:
MODEL      = "l2thinker"
REPO_DIR   = "/content/tinyaya-quant-safety"           # re-declared: the restart after Part 0 clears Part 0's variables
DATA_ROOT  = "/content/tinyaya-eval"
DRIVE_DIR  = "/content/drive/MyDrive/tinyaya-eval"      # "" disables the mirror
MAX_NEW, N_CTX = 8192, 10240

import os
from google.colab import userdata
os.environ.update({"TINYAYA_MODEL": MODEL, "TINYAYA_ROOT": DATA_ROOT, "TINYAYA_DRIVE_DIR": DRIVE_DIR,
                   "TINYAYA_MAX_NEW": str(MAX_NEW), "TINYAYA_N_CTX": str(N_CTX), "HF_TOKEN": userdata.get("HF_TOKEN")})
os.makedirs(f"{DATA_ROOT}/logs", exist_ok=True)
import importlib, tinyaya_eval.config as config, tinyaya_eval.common as common, tinyaya_eval.l2 as l2
importlib.reload(config); importlib.reload(common); importlib.reload(l2)
print(f"model={config.MODEL_NAME} repo={config.HF_REPO} max_new={config.MAX_NEW_TOKENS} n_ctx={config.N_CTX} root={config.ROOT}")
GPU = l2.gpu_name(); print("GPU:", GPU)

In [ ]:
if DRIVE_DIR:
    from google.colab import drive
    drive.mount("/content/drive")
    print(f"restored {common.restore_runs()} file(s) from the mirror")
    import shutil, pathlib   # runs_l2 is mirrored file-by-file by l2.gen; restore it too
    src = pathlib.Path(DRIVE_DIR) / "runs_l2"
    if src.exists():
        shutil.copytree(src, pathlib.Path(DATA_ROOT) / "runs_l2", dirs_exist_ok=True); print("restored runs_l2/")

In [ ]:
# 1b · Restore from Drive instead of reconverting (skip on the very first run; Part 2 then creates the GGUFs)
!mkdir -p /content/tinyaya-eval/models/gguf /content/tinyaya-eval/calib /content/tinyaya-eval/runs_l2
!cp -nv /content/drive/MyDrive/tinyaya-eval/models/gguf/l2thinker*.gguf /content/tinyaya-eval/models/gguf/ 2>/dev/null || echo "no GGUFs on Drive yet"
!cp -rn /content/drive/MyDrive/tinyaya-eval/calib/. /content/tinyaya-eval/calib/ 2>/dev/null || true
!cp -rn /content/drive/MyDrive/tinyaya-eval/runs_l2/. /content/tinyaya-eval/runs_l2/ 2>/dev/null || true
!ls -la /content/tinyaya-eval/models/gguf/ 2>/dev/null | grep gguf || echo "no local GGUFs: run Part 2"

In [ ]:
# 1.x · CUDA fix (Colab ships CUDA 13; the llama-cpp-python wheel wants CUDA 12). Run every session, after Part 1.
!pip install -q nvidia-cuda-runtime-cu12 nvidia-cublas-cu12
import os, glob, site, subprocess, sys
libs = [d for sp in site.getsitepackages() for d in glob.glob(f"{sp}/nvidia/*/lib")]
os.environ["LD_LIBRARY_PATH"] = ":".join(libs + [os.environ.get("LD_LIBRARY_PATH", "")])
r = subprocess.run([sys.executable, "-c", "import llama_cpp; print('llama_cpp loads OK')"], capture_output=True, text=True)
print(r.stdout or r.stderr[-800:])

In [ ]:
# fasttext for Python 3.13: try the wheel, else build 0.9.3 from source (~2 min)
import subprocess, sys, importlib
try:
    import fasttext
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "fasttext-wheel"], capture_output=True, text=True)
    importlib.invalidate_caches()
    try:
        import fasttext
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pybind11", "setuptools", "wheel"], check=True)
        r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation", "fasttext==0.9.3"], capture_output=True, text=True)
        if r.returncode != 0: print(r.stderr[-2000:])
        importlib.invalidate_caches()
        import fasttext
print("fasttext OK:", fasttext.__file__)

In [ ]:
# update l2.py from the repo, check it's the new version, reload
!cd {REPO_DIR} && git pull -q
import os, importlib
from pathlib import Path
s = (Path(REPO_DIR) / "tinyaya_eval" / "l2.py").read_text()
assert "prompt_seed" in s and "_LID.f.predict" in s, "old l2.py: push the updated file to GitHub first (or copy it in by hand)"
os.environ["TINYAYA_DECODING"] = "greedy"          # this notebook compares trimmed vs untrimmed: greedy
importlib.reload(l2)
print("decoding:", l2.DECODING, "|", l2.lid("বাংলাদেশের রাজধানী ঢাকা এবং এটি একটি বড় শহর"))   # greedy | bn

## 2 · Artifacts

HF safetensors → f16 GGUF → q8_0 / q4_k_m / q4_0 with the pinned llama.cpp. Same Cohere2 architecture as Tiny Aya (36 layers, GQA 16/4, tied 262k embedding), so the converter at this pin should handle it; if `convert` fails, the pin must move and **all four precisions move together** (then not comparable to the safety study). Expected sizes ≈ 6.7 / 3.6 / 2.1 / 2.0 GB. The resolved sha printed here is the pin for the write-up (the repo was last modified 2026-09-25).

**Skip this part if the GGUFs were restored from Drive in 1b.** Three deviations from the stock converter are applied here and must be reported in any write-up: (1) pre-tokenizer hash alias → `tiny_aya` (tokenizer verified identical to tiny-aya-global except add-EOS on encode); (2) `worker-*.safetensors` shards exposed as `model-*` (otherwise the converter silently writes a tensor-less 12 MB GGUF); (3) `--no-lazy` (lazy mode crashes on the all-zero LayerNorm bias check).

In [ ]:
!cd /content/llama.cpp && cmake -B build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF -DLLAMA_BUILD_TESTS=OFF -DLLAMA_BUILD_EXAMPLES=OFF -DLLAMA_BUILD_SERVER=OFF > /dev/null 2>&1 && cmake --build build --target llama-quantize -j$(nproc) 2>&1 | tail -1
# 2 · Convert L2-Thinker to f16 with the three documented fixes (pre-tokenizer alias, shard names, eager load), then quantize
import os, sys, json, shutil, subprocess, inspect, importlib
from pathlib import Path
from gguf import GGUFReader
HF, GGUF = Path("/content/tinyaya-eval/models/hf/l2thinker"), Path("/content/tinyaya-eval/models/gguf")
GGUF.mkdir(parents=True, exist_ok=True); F16 = GGUF / "l2thinker-f16.gguf"

def prep(step):
    r = subprocess.run([sys.executable, "-m", "tinyaya_eval.prepare", step], env=os.environ.copy(), cwd=REPO_DIR)
    assert r.returncode == 0, f"prepare {step} failed; see /content/tinyaya-eval/logs/"
if not (HF / "config.json").exists():
    prep("download")

# fix 1: pre-tokenizer hash alias (tokenizer verified identical to tiny-aya-global in the quant session)
L2_HASH = "9e079518a3ea5a3fe148bd4a570f4962db5db05a20e0cf74632de683e054d5b7"
sys.path.insert(0, "/content/llama.cpp")
base = importlib.import_module("conversion.base")
owner = next(c for _, c in inspect.getmembers(base, inspect.isclass) if "get_vocab_base_pre" in c.__dict__)
f = Path(inspect.getsourcefile(owner)); s = f.read_text()
if L2_HASH not in s:
    fn = s.index("def get_vocab_base_pre"); at = s.index("        res = None\n", fn) + len("        res = None\n")
    s = s[:at] + f'        if chkhsh == "{L2_HASH}":\n            # tiny-aya-l2-thinker: identical to tiny-aya-global except add-EOS on encode\n            res = "tiny_aya"\n' + s[at:]
    f.write_text(s); print("pre-tokenizer alias added")

# fix 2: expose worker-*.safetensors under model-*.safetensors names (symlinks) and rewrite the index
workers = sorted(HF.glob("worker-*.safetensors")); N = len(workers)
rename = {w.name: f"model-{i+1:05d}-of-{N:05d}.safetensors" for i, w in enumerate(workers)}
for old, new in rename.items():
    link = HF / new
    if link.is_symlink() or link.exists(): link.unlink()
    link.symlink_to(HF / old)
idx, orig = HF / "model.safetensors.index.json", HF / "model.safetensors.index.json.orig"
if not orig.exists(): shutil.copy(idx, orig)
j = json.loads(orig.read_text()); j["weight_map"] = {k: rename.get(v, v) for k, v in j["weight_map"].items()}
idx.write_text(json.dumps(j, indent=2)); print(f"{N} shards symlinked, {len(j['weight_map'])} tensors in index")

# fix 3: eager conversion (lazy mode crashes on the all-zero LayerNorm bias check)
F16.unlink(missing_ok=True)
r = subprocess.run([sys.executable, "/content/llama.cpp/convert_hf_to_gguf.py", str(HF), "--outfile", str(F16), "--outtype", "f16", "--no-lazy"],
                   capture_output=True, text=True)
print((r.stdout + r.stderr)[-1500:]); assert r.returncode == 0, "convert failed"
nt = len(GGUFReader(str(F16)).tensors); print(f"f16: {F16.stat().st_size/1e9:.2f} GB, {nt} tensors")
assert nt > 0 and F16.stat().st_size > 6e9
prep("quantize")
prep("inventory")
for p in ("q8_0", "q4_k_m", "q4_0"):
    q = GGUF / f"l2thinker-{p}.gguf"
    print(f"{p}: {q.stat().st_size/1e9:.2f} GB, {len(GGUFReader(str(q)).tensors)} tensors")

r = subprocess.run([sys.executable, "-c", "from llama_cpp import Llama; import sys; Llama(model_path=sys.argv[1], n_ctx=512, n_gpu_layers=-1, verbose=False); print('LOAD OK')", str(F16)],
                   capture_output=True, text=True, env=os.environ.copy())
print(r.stdout.strip() or r.stderr[-800:])
print("pinned revision:", common.revision())

In [ ]:
PRECISIONS = ["f16", "q4_k_m"]

## 3 · Preflight gate (thinking mode)

For every GGUF: the template carries the trigger and ends with `<|START_THINKING|>`; the four think delimiters are single tokens; the stop set has `<|END_RESPONSE|>` and **not** `<|END_THINKING|>`; a probe closes its think block, produces an answer, stops, and is deterministic. Nothing runs past a failure (a missed stop id would send every trace to the cap and make truncation meaningless — the trimming study's fresh-prompt landmine).

In [ ]:
import subprocess, sys
fails = []
for prec in PRECISIONS:
    r = subprocess.run([sys.executable, "-m", "tinyaya_eval.l2", "probe", "--precision", prec], capture_output=True, text=True, env=os.environ.copy(), cwd=REPO_DIR)
    print(r.stdout, end="")
    if r.returncode != 0:
        fails.append(prec); print("  stderr:", r.stderr[-1500:])
assert not fails, f"preflight failed: {fails}"

In [ ]:
# Memory-aware runner used by every generation cell. One process per job; a job starts only when its VRAM
# estimate fits beside the live ones. Resumable (gen skips ids that already have rows). Logs under logs/.
import os, sys, time, subprocess
from pathlib import Path
r = subprocess.run(["pgrep", "-fa", "[t]inyaya_eval.l2 gen"], capture_output=True, text=True)   # [t] stops pgrep matching itself
assert not r.stdout.strip(), f"old jobs still running, kill them first:\n{r.stdout}"
BUDGET_GB = 36.0 if "A100" in l2.gpu_name() else 14.0            # A100 40 GB; T4 15 GB
COST = {"f16": 8.0, "q8_0": 5.0, "q4_k_m": 3.5, "q4_0": 3.5}   # weights + KV at n_ctx 10240
LOGS = Path(DATA_ROOT) / "logs"; LOGS.mkdir(exist_ok=True)
def run_jobs_budget(jobs, tag):
    queue, live, rc = list(jobs), {}, {}
    while queue or live:
        used = sum(COST[j["precision"]] for _, j in live.values())
        for j in list(queue):
            if used + COST[j["precision"]] <= BUDGET_GB:
                queue.remove(j); used += COST[j["precision"]]
                m = j.get("model", config.MODEL_NAME); v = j.get("variant", "L2")
                name = f"{m}_{j['dataset']}_{v}_{j['precision']}"
                cmd = [sys.executable, "-m", "tinyaya_eval.l2", "gen", "--precision", j["precision"], "--dataset", j["dataset"],
                       "--langs", ",".join(j["langs"]), "--n", str(j["n"]), "--variant", v, "--tag", tag]
                live[name] = (subprocess.Popen(cmd, stdout=open(LOGS / f"gen_{name}.log", "a"), stderr=subprocess.STDOUT,
                                               env={**os.environ, "TINYAYA_MODEL": m}, cwd=REPO_DIR), j)
                print(time.strftime("%H:%M"), "started", name, flush=True); time.sleep(60)
        time.sleep(30)
        for name, (p, _) in list(live.items()):
            if p.poll() is not None:
                rc[name] = p.returncode; del live[name]
                print(time.strftime("%H:%M"), "finished", name, "rc", rc[name], "| live", len(live), "| queued", len(queue), flush=True)
    bad = [k for k, v in rc.items() if v != 0]
    assert not bad, f"failed: {bad} (see logs/gen_<name>.log)"
print("runner ready | GPU", l2.gpu_name(), "| budget", BUDGET_GB, "GB")
# Known quirk: an Aya job that has nothing left to generate ("0 done, 0 to go" on resume) does not exit, because the
# streaming dataset iterator leaves non-daemon threads alive. Kill just that PID from the Terminal; its rows are complete.

In [ ]:
import json, statistics
from pathlib import Path
import pandas as pd
def rows_df(tag, model=None):
    d = Path(DATA_ROOT) / "runs_l2" / (model or config.MODEL_NAME) / tag
    recs = [json.loads(l) for p in sorted(d.glob("*.jsonl")) for l in p.read_text(encoding="utf-8").splitlines() if l.strip()]
    df = pd.DataFrame(recs)
    if len(df):
        df["tier"] = df.lang.map(l2.LANG_TIER)
    return df
def add_lid(df):
    if "trace_lang" not in df:
        df["trace_lang"] = [l2.lid(t) if isinstance(t, str) and t.strip() else None for t in df.trace]
        df["l2"] = df.trace_lang == df.lang
    return df
def same_gpu(df):
    g = df.gpu.unique(); assert len(g) == 1, f"mixed GPUs {g}: every paired comparison must come from one GPU type/session"; return g[0]
print("loaders ready")

## 10 · Trim setup: names, paths, tools

The CPU build of `llama-quantize` and `llama-bench` is needed for Part 14 (sizes and CPU decode speed, the phone proxy). It takes ~5 min and is wiped by a disconnect.

In [ ]:
# 2.x · CPU build of llama-quantize and llama-bench at the pinned commit, copied to build/bin
import subprocess
from pathlib import Path
LLAMA = Path("/content/llama.cpp")
r = subprocess.run(f"cd {LLAMA} && cmake -B build-cpu -DGGML_CUDA=OFF -DLLAMA_CURL=OFF -DLLAMA_BUILD_TESTS=OFF -DLLAMA_BUILD_SERVER=OFF "
                   f"&& cmake --build build-cpu --target llama-quantize llama-bench -j$(nproc)", shell=True, capture_output=True, text=True)
print((r.stdout + r.stderr)[-1500:])
assert r.returncode == 0, "build failed (see output above)"
(LLAMA / "build" / "bin").mkdir(parents=True, exist_ok=True)
for tool in ("llama-quantize", "llama-bench"):
    src, dst = LLAMA / "build-cpu" / "bin" / tool, LLAMA / "build" / "bin" / tool
    subprocess.run(["cp", "-f", str(src), str(dst)], check=True)
    print(tool, "->", dst, dst.is_file())
print("pin:", subprocess.run(["git", "-C", str(LLAMA), "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())

In [ ]:
import os, sys, json, hashlib, subprocess, importlib, shutil
from pathlib import Path
REPO  = Path(REPO_DIR); ROOT = Path(DATA_ROOT); LLAMA = Path("/content/llama.cpp")
BASE, TRIM, CTRL = "l2thinker", "l2thinker_trim", "l2thinker_ctrl"      # registered by `l2 patch`
GGUF, TRIMDIR, LOGS = ROOT / "models" / "gguf", ROOT / "calib" / "trim" / BASE, ROOT / "logs"
for d in (TRIMDIR / "corpus", TRIMDIR / "heldout", LOGS): d.mkdir(parents=True, exist_ok=True)
assert (REPO / "tinyaya_eval" / "vocabtrim.py").exists(), "copy vocabtrim.py into tinyaya_eval/ and push it"
F16 = GGUF / f"{BASE}-f16.gguf"; assert F16.exists(), f"{F16} missing: run Part 2"
QUANTIZE_BIN, BENCH_BIN = LLAMA / "build" / "bin" / "llama-quantize", LLAMA / "build" / "bin" / "llama-bench"
assert QUANTIZE_BIN.is_file() and BENCH_BIN.is_file()
assert subprocess.run(["git", "-C", str(LLAMA), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip() == config.LLAMA_CPP_PIN
# sanity for the "preserved by construction" argument (handoff §5): tied embedding, n_embd % 256 == 0, byte-level BPE
from llama_cpp import Llama
vt = Llama(model_path=str(F16), vocab_only=True, verbose=False)
md = vt.metadata; print({k: md.get(k) for k in ("tokenizer.ggml.model", "cohere2.embedding_length", "general.architecture")})
assert md.get("tokenizer.ggml.model") == "gpt2", "vocabtrim's ancestor closure assumes byte-level BPE with merges"
del vt
print("base:", BASE, "| trim:", TRIM, "| f16:", f"{F16.stat().st_size/1e9:.2f} GB")

In [ ]:
# 10b · Reproduce the shipped keep file instead of rebuilding the corpus (then SKIP Part 11).
# Put keep/l2thinker_trim_keep.json from the project repo on Drive under tinyaya-eval/calib/trim/l2thinker/, or upload it here.
import hashlib, shutil
src = Path("/content/drive/MyDrive/tinyaya-eval/calib/trim/l2thinker/l2thinker_trim_keep.json")
if src.exists():
    shutil.copy(src, TRIMDIR / f"{TRIM}_keep.json")
    j = json.loads((TRIMDIR / f"{TRIM}_keep.json").read_text())
    print("keep file restored:", j["stats"]["kept"], "of", j["stats"]["vocab"], "tokens; sha", j["stats"]["keep_sha256"][:16])
    assert j["stats"]["keep_sha256"].startswith("bfadfaf070959033"), "not the keep file evaluated in the write-up"
else:
    print("no shipped keep file found: run Part 11 to build the corpus, then Part 12 selects a new keep set")

## 11 · Selection corpus

FineWeb-2 + Aya Collection per region language + FineWeb-Edu English + code (as in the Fire trim), **plus** the region's translated reasoning traces (question + thinking + output) from the released MR data, train split. Held-out: FineWeb-2 test, English, and the MR **test** split (reasoning traces the selection never saw). MultiJail, MGSM, Macaron and the Aya skip-200k eval slice are excluded.

**Skip if 10b restored the keep file.**

In [ ]:
from datasets import load_dataset, get_dataset_config_names
REGION = {  # Fire = South Asia (Tiny Aya paper Table 1). Earth/Water: change these maps.
    "bn": "ben_Beng", "hi": "hin_Deva", "mr": "mar_Deva", "gu": "guj_Gujr", "pa": "pan_Guru",
    "ta": "tam_Taml", "te": "tel_Telu", "ne": "npi_Deva", "ur": "urd_Arab"}
SCRIPTS = "BENGALI,DEVANAGARI,GUJARATI,GURMUKHI,TAMIL,TELUGU,ARABIC"     # every token entirely in these scripts is kept (v2 rule)
MR_LANGS = [l for l in REGION if l in l2.L2_45]                          # reasoning data exists only for the 45 L2 languages
MB_WEB, MB_AYA, MB_EN, MB_CODE, MB_HELD, MB_MR = 40, 10, 80, 40, 3, 20

def stream_to(path, ds, field, mb):
    if path.exists() and path.stat().st_size >= mb * 1e6 * 0.9: return path.stat().st_size
    n = 0
    with open(path, "w", encoding="utf-8") as f:
        for row in ds:
            t = (row.get(field) or "").strip() if isinstance(row, dict) else ""
            if t: f.write(t + "\n"); n += len(t.encode())
            if n >= mb * 1e6: break
    return n

aya_avail = set(get_dataset_config_names("CohereForAI/aya_collection_language_split"))
for lang, cfg_ in REGION.items():
    n = stream_to(TRIMDIR / "corpus" / f"web_{lang}.txt", load_dataset("HuggingFaceFW/fineweb-2", name=cfg_, split="train", streaming=True), "text", MB_WEB)
    h = stream_to(TRIMDIR / "heldout" / f"web_{lang}.txt", load_dataset("HuggingFaceFW/fineweb-2", name=cfg_, split="test", streaming=True), "text", MB_HELD)
    a = 0
    if l2.AYA_NAMES.get(lang) in aya_avail:
        ds = load_dataset("CohereForAI/aya_collection_language_split", l2.AYA_NAMES[lang], split="train", streaming=True)
        a = stream_to(TRIMDIR / "corpus" / f"aya_{lang}.txt", ({"text": f"{r.get('inputs','')}\n{r.get('targets','')}"} for r in ds), "text", MB_AYA)
    print(f"{lang}: web {n/1e6:.0f} MB, aya {a/1e6:.0f} MB, heldout {h/1e6:.1f} MB")
# reasoning traces: the distribution a trimmed L2-Thinker must reproduce. Train split -> corpus; test split -> held-out.
for lang in MR_LANGS:
    for split, dst, mb in (("train", TRIMDIR / "corpus" / f"mr_{lang}.txt", MB_MR), ("test", TRIMDIR / "heldout" / f"mr_{lang}.txt", MB_HELD)):
        try:
            ds = load_dataset("CohereLabs/tiny-aya-l2-thinker-multilingual-reasoning", data_files=f"data/{lang}/{split}.parquet", split="train", streaming=True)
            n = stream_to(dst, ({"text": f"{r['question']}\n<|START_THINKING|>{r['thinking']}<|END_THINKING|><|START_RESPONSE|>{r['output']}<|END_RESPONSE|>"} for r in ds), "text", mb)
            print(f"mr {lang} {split}: {n/1e6:.1f} MB")
        except Exception as e:
            print(f"mr {lang} {split}: skipped ({str(e)[:80]})")
print("en:", stream_to(TRIMDIR / "corpus" / "web_en.txt", load_dataset("HuggingFaceFW/fineweb-edu", name="sample-10BT", split="train", streaming=True), "text", MB_EN) / 1e6, "MB")
print("code:", stream_to(TRIMDIR / "corpus" / "code.txt", load_dataset("codeparrot/codeparrot-clean-valid", split="train", streaming=True), "content", MB_CODE) / 1e6, "MB")
stream_to(TRIMDIR / "heldout" / "web_en.txt", load_dataset("HuggingFaceFW/fineweb-edu", name="sample-10BT", split="train", streaming=True).skip(20000), "text", MB_HELD)
# held-out: the harness's own prompt material (never in the corpus)
(TRIMDIR / "heldout" / "mgsm_bn.txt").write_text("\n".join(it["prompt"] for it in l2.load_items("mgsm", ["bn"], 250)), encoding="utf-8")
(TRIMDIR / "heldout" / "mgsm_en.txt").write_text("\n".join(it["prompt"] for it in l2.load_items("mgsm", ["en"], 250)), encoding="utf-8")
print("corpus MB:", sum(p.stat().st_size for p in (TRIMDIR / "corpus").glob("*.txt")) / 1e6)

## 12 · Select, apply, pin

v2 rule: corpus tokens + special/control + 256 byte tokens + every no-letter token + every token entirely in the region's scripts + BPE-ancestor closure. `apply` aborts if any `*_token_id` (incl. the think delimiters) would be removed. The revision string is base sha + keep hash, so run identities never collide with the base model's.

In [ ]:
KEEP = TRIMDIR / f"{TRIM}_keep.json"; OUT16 = GGUF / f"{TRIM}-f16.gguf"
if not KEEP.exists():
    subprocess.run([sys.executable, "-m", "tinyaya_eval.vocabtrim", "select", "--src", str(F16), "--corpus", str(TRIMDIR / "corpus" / "*.txt"),
                    "--out", str(KEEP), "--keep-no-letters", "--keep-scripts", SCRIPTS], check=True, cwd=str(REPO))
stats = json.loads(KEEP.read_text())["stats"]; print({k: stats[k] for k in ("vocab", "used_in_corpus", "kept", "kept_pct", "keep_sha256")})
keep = set(json.loads(KEEP.read_text())["keep"])
vt = Llama(model_path=str(F16), vocab_only=True, verbose=False)
for s in l2.THINK.values():
    tid = vt.tokenize(s.encode(), add_bos=False, special=True)[0]; assert tid in keep, f"{s} ({tid}) not in keep set"
del vt; print("think delimiters kept")
if not OUT16.exists():
    subprocess.run([sys.executable, "-m", "tinyaya_eval.vocabtrim", "apply", "--src", str(F16), "--keep", str(KEEP), "--dst", str(OUT16)], check=True, cwd=str(REPO))
print(json.loads(Path(str(OUT16) + ".trim.json").read_text()))
base_rev = common.revision_file(BASE).read_text().strip()
rf = common.revision_file(TRIM); rf.write_text(f"{base_rev}+trim{stats['keep_sha256']}"); common.mirror(rf)
for p in (KEEP, Path(str(OUT16) + ".trim.json")): common.mirror(p)
print("revision:", rf.read_text())

## 13 · Tokenization drift on held-out text (web, English, reasoning traces, MGSM prompts)

In [ ]:
rows = []
for p in sorted((TRIMDIR / "heldout").glob("*.txt")):
    r = subprocess.run([sys.executable, "-m", "tinyaya_eval.vocabtrim", "verify", "--src", str(F16), "--dst", str(OUT16), "--texts", str(p)], capture_output=True, text=True, cwd=str(REPO))
    j = json.loads(r.stdout[r.stdout.rfind("{"):]) if "{" in r.stdout else {}
    rows.append((p.stem, j.get("lines"), j.get("lines_tokenized_identically_pct"), j.get("fertility_increase_pct")))
print(f"{'heldout':16s} {'lines':>7s} {'identical %':>12s} {'fertility +%':>13s}")
for name, n, ident, fert in rows:
    print(f"{name:16s} {n or 0:7d} {ident or 0:12.2f} {fert or 0:13.2f}{'  <-- over 1%' if (fert or 0) > 1.0 else ''}")
print("\nrule: every in-region language, INCLUDING mr_<lang> reasoning traces, under 1% fertility increase")

## 14 · Quantize the trimmed model, sizes, CPU decode speed (phone proxy)

In [ ]:
env_trim = {**os.environ, "TINYAYA_MODEL": TRIM, "TINYAYA_CALIB": ""}
subprocess.run([sys.executable, "-m", "tinyaya_eval.prepare", "quantize"], check=True, env=env_trim, cwd=str(REPO))
print(f"{'file':32s} {'GB':>6s} {'vs base':>8s}")
for prec in ("f16", "q8_0", "q4_k_m", "q4_0"):
    a, b = GGUF / f"{BASE}-{prec}.gguf", GGUF / f"{TRIM}-{prec}.gguf"
    if a.exists() and b.exists(): print(f"{b.name:32s} {b.stat().st_size/1e9:6.2f} {100*(b.stat().st_size/a.stat().st_size-1):7.1f}%")
def bench(model, ngl, threads, n=100):
    r = subprocess.run([str(BENCH_BIN), "-m", str(model), "-p", "100", "-n", str(n), "-ngl", str(ngl), "-t", str(threads), "-r", "5", "-o", "json"], capture_output=True, text=True)
    out = json.loads(r.stdout[r.stdout.find("["):]) if "[" in r.stdout else []
    return {d.get("test", d.get("n_gen", "?")): round(d["avg_ts"], 1) for d in out} or r.stderr[-400:]
for name in (BASE, TRIM):
    print(f"cpu {name}-q4_k_m:", bench(GGUF / f"{name}-q4_k_m.gguf", 0, os.cpu_count()))
print("\nrule: q4_k_m >= 10% smaller and CPU tg100 >= 5% faster; report tg only (pp on 2 cores is noise); add peak RSS on a real device")

## 15 · Free generation, same session, same GPU: trimmed vs control

The untrimmed model is regenerated in this session as `l2thinker_ctrl` (symlinked GGUFs, revision `<sha>+ctrl`), so trimmed and control rows come from the same code, session and GPU type. Greedy decoding throughout (identity is the test). Rows land in `runs_l2/<model>/<tag>/` and are mirrored to Drive as they are written.

Jobs below reproduce the write-up's tags: `trim_a100` (round 1), `trim_a100_rerun` (E0, control second pass), `trim_a100_r2` (E1/E2), `trim_a100_r2x` (exploratory MGSM sw). Run one runner cell at a time; each resumes if rerun.

In [ ]:
# 15a · Control links: the untrimmed GGUFs under a second name. Recreate every session (they live in /content).
rf = common.revision_file(CTRL); rf.write_text(common.revision_file(BASE).read_text().strip() + "+ctrl"); common.mirror(rf)
for prec in ("f16", "q4_k_m"):
    src, link = GGUF / f"{BASE}-{prec}.gguf", GGUF / f"{CTRL}-{prec}.gguf"
    assert src.exists(), f"missing base {src}"
    if link.is_symlink() or link.exists(): link.unlink()
    link.symlink_to(src); assert link.exists(), f"dangling link {link}"
    print(link.name, "->", link.resolve().name)

In [ ]:
# 15b · Save the trimmed GGUFs to Drive once (rebuilding takes ~15 min; reconverting the base ~40 min)
!cp -nv /content/tinyaya-eval/models/gguf/l2thinker*.gguf /content/tinyaya-eval/models/gguf/*.trim.json /content/drive/MyDrive/tinyaya-eval/models/gguf/

In [ ]:
# 15c · Round 1 (tag trim_a100): MGSM bn 250, Aya bn+en 100, f16 and q4_k_m, trimmed and control. ~4.5 h on A100.
assert l2.DECODING == "greedy"
jobs  = [{"model": m, "precision": p, "dataset": "mgsm", "langs": ["bn"], "n": 250} for m in (TRIM, CTRL) for p in ("f16", "q4_k_m")]
jobs += [{"model": m, "precision": p, "dataset": "aya",  "langs": ["bn", "en"], "n": 100} for m in (TRIM, CTRL) for p in ("f16", "q4_k_m")]
run_jobs_budget(jobs, tag="trim_a100")

In [ ]:
# 15d · E0 noise floor (tag trim_a100_rerun): the control a second time, same config. ~4.5 h. Must be the same GPU type as round 1.
jobs  = [{"model": CTRL, "precision": p, "dataset": "mgsm", "langs": ["bn"], "n": 250} for p in ("f16", "q4_k_m")]
jobs += [{"model": CTRL, "precision": p, "dataset": "aya",  "langs": ["bn", "en"], "n": 100} for p in ("f16", "q4_k_m")]
run_jobs_budget(jobs, tag="trim_a100_rerun")

In [ ]:
# 15e · Round 2 E1/E2 (tag trim_a100_r2): MGSM te 250; Aya hi, gu, ta (in region) and sw (out of region) 100 each; f16. ~3 h.
for l in ("hi", "gu", "ta", "sw"): assert len(l2.load_items("aya", [l], 5)) == 5, l
jobs  = [{"model": m, "precision": "f16", "dataset": "mgsm", "langs": ["te"], "n": 250} for m in (TRIM, CTRL)]
jobs += [{"model": m, "precision": "f16", "dataset": "aya",  "langs": ["hi", "gu", "ta", "sw"], "n": 100} for m in (TRIM, CTRL)]
run_jobs_budget(jobs, tag="trim_a100_r2")

In [ ]:
# 15f · Exploratory (tag trim_a100_r2x): MGSM sw on the South Asia trim, the objective out-of-region cost. ~3 h.
jobs = [{"model": m, "precision": "f16", "dataset": "mgsm", "langs": ["sw"], "n": 250} for m in (TRIM, CTRL)]
run_jobs_budget(jobs, tag="trim_a100_r2x")

In [ ]:
# 15g · Back up rows after every runner (gen mirrors rows itself; this is belt and braces)
!mkdir -p /content/drive/MyDrive/tinyaya-eval/runs_l2 && cp -ru /content/tinyaya-eval/runs_l2/l2thinker_trim /content/tinyaya-eval/runs_l2/l2thinker_ctrl /content/drive/MyDrive/tinyaya-eval/runs_l2/
!wc -l /content/tinyaya-eval/runs_l2/l2thinker_*/trim_a100*/*.jsonl

In [ ]:
# 15h · Readout for one tag: identity, metric deltas, exact McNemar, pre-registered checks, divergence attribution (retokenization).
import json, math, numpy as np, pandas as pd
from llama_cpp import Llama
TAG = "trim_a100"                                   # trim_a100 | trim_a100_r2 | trim_a100_r2x
IN_REGION = {"bn", "en", "hi", "gu", "ta", "te", "mr", "pa", "ne", "ur"}
ct, tr = add_lid(rows_df(TAG, CTRL)), add_lid(rows_df(TAG, TRIM))
for d in (ct, tr):
    m = d.dataset == "mgsm"
    d.loc[m, "correct"] = [l2.mgsm_correct(a, g) for a, g in zip(d.loc[m, "answer"], d.loc[m, "gold"])]
both = pd.concat([ct, tr]); same_gpu(both); assert set(both.decoding) == {"greedy"}
KEY = ["dataset", "precision", "lang", "id"]
assert not ct.duplicated(KEY).any() and not tr.duplicated(KEY).any(), "duplicate rows"
p = ct.merge(tr, on=KEY, suffixes=("_c", "_t"), validate="1:1"); assert len(p) == len(ct) == len(tr), (len(p), len(ct), len(tr))
p["same"] = (p.trace_c == p.trace_t) & (p.answer_c == p.answer_t)
def mcnemar_p(a, b):
    n01, n10 = int((~a & b).sum()), int((a & ~b).sum()); n = n01 + n10
    return 1.0 if n == 0 else min(1.0, 2 * sum(math.comb(n, k) for k in range(min(n01, n10) + 1)) / 2 ** n)
rows = []
for (ds, prec, lang), g in p.groupby(["dataset", "precision", "lang"]):
    r = dict(dataset=ds, prec=prec, lang=lang, n=len(g), identity=100 * g.same.mean(),
             d_trunc=100 * (g.truncated_t.mean() - g.truncated_c.mean()), d_L2=100 * (g.l2_t.mean() - g.l2_c.mean()),
             d_loop=g.doomloop_t.mean() - g.doomloop_c.mean(), trunc_c=100 * g.truncated_c.mean(), L2_c=100 * g.l2_c.mean(), loop_c=g.doomloop_c.mean())
    if ds == "mgsm":
        a, b = g.correct_c.astype(bool).values, g.correct_t.astype(bool).values
        r.update(acc_c=100 * a.mean(), acc_t=100 * b.mean(), d_acc=100 * (b.mean() - a.mean()), mcnemar_p=mcnemar_p(a, b))
    rows.append(r)
res = pd.DataFrame(rows); pd.set_option("display.width", 220); print(res.round(3).to_string(index=False))
e = res[res.lang.isin(IN_REGION)]
print("\nPre-registered checks (in-region groups):")
f16 = e[e.prec == "f16"]; print(f"  f16 identity >= 98%: {f16.identity.round(2).tolist()} ->", "PASS" if (f16.identity >= 98).all() else "FAIL")
for col, tol in (("d_trunc", 1), ("d_L2", 1), ("d_acc", 1), ("d_loop", 0.01)):
    v = e[col].dropna(); print(f"  {col} within ±{tol}: {v.round(3).tolist()} ->", "PASS" if (v.abs() <= tol).all() else "FAIL")
mp = e.mcnemar_p.dropna() if "mcnemar_p" in e else pd.Series(dtype=float); print(f"  McNemar n.s.: {mp.round(3).tolist()} ->", "PASS" if (mp > 0.05).all() else "FAIL")
# attribution by retokenization (approximate; replay in 15i is authoritative)
keep = set(json.loads((TRIMDIR / f"{TRIM}_keep.json").read_text())["keep"])
vt = Llama(model_path=str(F16), vocab_only=True, verbose=False)
def removed_before(text, char_idx):
    target, pos, out = len(text[:char_idx].encode()), 0, []
    for t in vt.tokenize(text.encode(), add_bos=False, special=True):
        pos += len(vt.detokenize([t]))
        if t not in keep: out.append(vt.detokenize([t]).decode("utf-8", "replace"))
        if pos > target: break
    return out
att = []
for _, r in p[~p.same].iterrows():
    f = "trace" if r.trace_c != r.trace_t else "answer"
    a, b = (r[f"{f}_c"] or ""), (r[f"{f}_t"] or "")
    if f == "answer": a, b = (r.trace_c or "") + a, (r.trace_t or "") + b
    k = next((i for i, (x, y) in enumerate(zip(a, b)) if x != y), min(len(a), len(b)))
    rm = removed_before(a, k); att.append(dict(lang=r.lang, prec=r.precision, id=r.id, field=f, at_char=k, removed=bool(rm), pieces=rm[-3:]))
att = pd.DataFrame(att)
if len(att):
    print(f"\n{len(att)} divergences; with a removed token at or before: {int(att.removed.sum())}")
    print(att.to_string(index=False))
else:
    print("\nno divergences")

In [ ]:
# 15i · Divergence replay (authoritative attribution). Rebuilds the prompt through the GGUF chat template WITH BOS,
# replays the control greedily through trace and answer, and reports the control's real token at the divergence.
from llama_cpp import Llama
from llama_cpp.llama_chat_format import Jinja2ChatFormatter
def build_prompt(M, row):
    items = l2.load_items(row.dataset, [row.lang], 250 if row.dataset == "mgsm" else 100)
    it = next((x for x in items if x.get("id") == row.id), None) or items[int(row.id.split(":")[-1])]
    fmt = Jinja2ChatFormatter(template=M.metadata["tokenizer.chat_template"],
                              eos_token=M.detokenize([M.token_eos()]).decode(), bos_token=M.detokenize([M.token_bos()]).decode())
    return fmt(messages=[{"role": "user", "content": it["prompt"]}]).prompt
def replay(rc_, rt_, prec):
    M = Llama(model_path=str(GGUF / f"{BASE}-{prec}.gguf"), n_ctx=8192, n_gpu_layers=-1, verbose=False)
    toks = M.tokenize(build_prompt(M, rc_).encode(), add_bos=True, special=True); assert toks[0] == M.token_bos()
    print(f"\n{rc_.id} {prec}: removed tokens in prompt:", [M.detokenize([t]).decode("utf-8", "replace") for t in toks if t not in keep])
    f = "trace" if rc_.trace != rt_.trace else "answer"
    a, b = rc_[f] or "", rt_[f] or ""
    k = next((i for i, (x, y) in enumerate(zip(a, b)) if x != y), min(len(a), len(b)))
    tr_t, an_t = (rc_.trace or "").encode(), (rc_.answer or "").encode()
    kb = len(a[:k].encode()) + (0 if f == "trace" else len(tr_t))
    mode, buf, path, result = "trace", b"", [], None
    for t in M.generate(toks, temp=0.0, repeat_penalty=1.0, reset=True):
        sp, piece = M.detokenize([t], special=True), M.detokenize([t])
        if b"<|END_THINKING|>" in sp: mode = "answer"; buf = b""; path.append(t); continue
        if not piece: path.append(t); continue
        target = tr_t if mode == "trace" else an_t
        cand = (buf + piece).strip()
        pos = len(cand) + (0 if mode == "trace" else len(tr_t))
        if pos > kb and (f == mode):
            result = ("control token at divergence", t, sp.decode("utf-8", "replace"), "REMOVED" if t not in keep else "in_keep"); break
        if not target.startswith(cand): result = (f"replay left the control's {mode}", buf[-60:], piece); break
        buf += piece; path.append(t)
        if len(path) > 3000: result = ("gave up",); break
    print(f"  field={f} char={k} replayed {len(path)} tokens | removed tokens in control's path:",
          [M.detokenize([u]).decode("utf-8", "replace") for u in path if u not in keep])
    print("  result:", result); del M
for _, r in att.iterrows():
    rc_ = ct[(ct.id == r.id) & (ct.precision == r.prec)].iloc[0]; rt_ = tr[(tr.id == r.id) & (tr.precision == r.prec)].iloc[0]
    replay(rc_, rt_, r.prec)

In [ ]:
# 15j · E0 noise floor: control vs control
c1, c2 = rows_df("trim_a100", CTRL), rows_df("trim_a100_rerun", CTRL)
q = c1.merge(c2, on=KEY, suffixes=("_1", "_2"), validate="1:1"); assert len(q) == 900, len(q)
q["same"] = (q.trace_1 == q.trace_2) & (q.answer_1 == q.answer_2)
print(q.groupby(["dataset", "precision", "lang"]).same.mean().mul(100).round(1))
print("control-vs-control identity:", round(100 * q.same.mean(), 2), "%"); print(q.loc[~q.same, KEY].to_string(index=False))

## 16 · KV-cache optimisation (roadmap C, exploratory)

Every log shows `llama_kv_cache_iswa: using full-size SWA cache`: 3 of 4 layers use a 4096-token sliding window but the cache is allocated at full `n_ctx`. Two knobs, measured for memory and output identity on 20 Aya prompts: `swa_full=False` (llama.cpp PR #13194) and KV quantization `type_k/type_v = q8_0` (V needs flash attention). Memory at 4K and 32K contexts. **Identity must be 100% at f16 for swa_full and is not expected for q8_0 KV (report it).** This is the setting that matters most for a 32K reasoning model on a phone.

The cell below separates the knobs (flash alone, q8 KV + flash, the combination with `swa_full=False`) and reports where outputs diverge. Divergence under greedy decoding is expected for any numerical change and is not a quality measure; lossy settings need the paired-metric protocol before they are cleared.

In [ ]:
import gc, subprocess, numpy as np, llama_cpp
from llama_cpp import Llama
import inspect
sig = inspect.signature(Llama.__init__).parameters
assert all(k in sig for k in ("swa_full", "flash_attn", "type_k")), "this llama-cpp-python build lacks a KV knob; the row would silently measure nothing"
def vram(): return int(subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"], capture_output=True, text=True).stdout.strip().splitlines()[0])
prompts = [it["prompt"] for it in l2.load_items("aya", ["bn"], 20)]
STOP = ["<|END_OF_TURN_TOKEN|>", "<|END_RESPONSE|>"]
def run(cfg_, n_ctx):
    kw = {k: v for k, v in cfg_.items() if k != "label"}
    base = vram(); M = Llama(model_path=str(GGUF / f"{BASE}-q4_k_m.gguf"), n_ctx=n_ctx, n_gpu_layers=-1, verbose=False, **kw)
    used = vram() - base; outs = []
    for p_ in prompts:
        M.reset(); r = M.create_chat_completion([{"role": "user", "content": p_}], max_tokens=512, temperature=0.0, stop=STOP)
        outs.append(r["choices"][0]["message"]["content"])
    del M; gc.collect(); return used, outs
def diverge_at(a, b): return next((i for i, (x, y) in enumerate(zip(a, b)) if x != y), min(len(a), len(b)))
configs = [{"label": "default"}, {"label": "swa_full=False", "swa_full": False}, {"label": "flash only", "flash_attn": True},
           {"label": "q8 KV + flash", "type_k": llama_cpp.GGML_TYPE_Q8_0, "type_v": llama_cpp.GGML_TYPE_Q8_0, "flash_attn": True},
           {"label": "q8 KV + flash + swa", "type_k": llama_cpp.GGML_TYPE_Q8_0, "type_v": llama_cpp.GGML_TYPE_Q8_0, "flash_attn": True, "swa_full": False}]
for n_ctx in (4096, 32768):
    ref = None
    for c in configs:
        try: used, outs = run(c, n_ctx)
        except Exception as e: print(f"n_ctx={n_ctx:5d} {c['label']:20s} FAILED: {str(e)[:120]}"); continue
        ref = ref or outs; d = [diverge_at(a, b) for a, b in zip(ref, outs)]
        ident = 100 * sum(x == len(a) for x, a in zip(d, ref)) / len(outs)
        print(f"n_ctx={n_ctx:5d} {c['label']:20s} VRAM +{used/1024:5.2f} GB  identity {ident:5.1f}%  diverge at char: median {int(np.median(d))}, min {min(d)}, of median len {int(np.median([len(a) for a in ref]))}")
print("expected: swa_full=False identical at both sizes and ~1.5 GB smaller at 32K; flash/q8 rows diverge (numerics), not a quality measure")

## 17 · Status

Results, pre-registered criteria, divergence analysis, external checks and next steps live in `docs/writeup.md` in the project repo. `notebooks/01_verify_from_rows.ipynb` reproduces every table from the shipped rows without a GPU.

In [ ]:
# Save session: base + trimmed GGUFs, keep file, rows, logs
import shutil
from pathlib import Path
D, L = Path("/content/drive/MyDrive/tinyaya-eval"), Path("/content/tinyaya-eval")
(D / "models" / "gguf").mkdir(parents=True, exist_ok=True)
for p_ in sorted((L / "models" / "gguf").glob("l2thinker*.gguf")) + sorted((L / "models" / "gguf").glob("*.trim.json")):
    if p_.is_symlink(): continue
    dst = D / "models" / "gguf" / p_.name
    if dst.exists() and dst.stat().st_size == p_.stat().st_size: continue
    print("copying", p_.name); shutil.copy2(p_, dst)
for sub in ("calib", "runs_l2", "logs"):
    if (L / sub).exists(): shutil.copytree(L / sub, D / sub, dirs_exist_ok=True)
print(sorted(p_.name for p_ in (D / "models" / "gguf").glob("l2thinker*")))